In [0]:
from pyspark.sql.functions import\
     current_timestamp\
    ,col\
    ,from_utc_timestamp
from pathlib import Path
from datetime import datetime

In [0]:
resource_path = '/Volumes/workspace/pj_sales/sales_resources/origin/'
processed_path = '/Volumes/workspace/pj_sales/sales_resources/processed/'

In [0]:
df_sales = spark.read\
    .format('csv')\
    .option('sep', ',')\
    .option('inferschema', 'false')\
    .option('header', 'true')\
    .load(resource_path)

In [0]:
df_sales = df_sales\
    .withColumn('ingestion_timestamp', from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo'))\
    .withColumn('source_file', col('_metadata.file_path'))
#uuid - Criação de hash único para cada registro (sugestão)

In [0]:
if any(Path(resource_path).iterdir()):
    df_sales.write\
        .format('delta')\
        .mode('append')\
        .option('mergeSchema', 'true')\
        .saveAsTable('workspace.pj_sales.tb_sales_bronze')
else:
    print('No new files to process')

In [0]:

processed_files = [f for f in dbutils.fs.ls(resource_path) if f.name.endswith('.csv')]
for file in processed_files:
  timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
  dbutils.fs.mv(file.path, processed_path + f'{timestamp}_' + file.name)
  print(f'File {file.name} moved to processed.')
print('All files processed!')